In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)

# import sys
# sys.path.append("/ihome/ylee/yiz133/Code/Data processing/functions")
import importlib
import functions.mdata_utils as mdata_utils
import functions.TCR_embedings as TCR_embedings

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import CCA

In [3]:
importlib.reload(TCR_embedings)

<module 'functions.TCR_embedings' from 'e:\\Python code\\Machine learning\\JupyterNote\\Bio_CRC\\Data processing\\functions\\TCR_embedings.py'>

In [4]:
# %cd "/ix1/ylee/Yifan_Zhang/Code_data/external"

In [5]:
%cd "./data/EAE/CCA"

e:\Python code\Machine learning\JupyterNote\Bio_CRC\Data processing\data\EAE\CCA


In [6]:
filename = "EAE_HV_then_merged_test293883"
mdata_ori = mu.read(filename + ".h5mu")

In [74]:
mdata = mdata_ori.copy()
mdata

MuData object with n_obs × n_vars = 125524 × 3001
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'cloned', 'condition', 'sample_id', 'set', 'state'
  2 modalities
    airr:	125524 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition'
      obsm:	'airr', 'chain_indices'
    gex:	125524 x 3001
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'

In [75]:
mdata['airr'].obs['GSE'] = mdata.obs['GSE']
mdata['airr'].obs['GSE'].value_counts()

GSE
LEE          43558
GSE182747    37164
GSE188320    22346
GSE293883    11097
GSE156718    10848
GSE178085      511
Name: count, dtype: int64

# Settings

In [76]:
# mdata_sets = {'all': mdata,
#             'cloned': mdata_cloned,
#             'one_in_each_clone': mdata_cloned_oneCell}
# mdata_sub = mdata_sets['cloned']

### Sets to use
subset_by_GSE = False
if subset_by_GSE == True:
    target_set = ['GSE188320', 'GSE178085']
    mdata = mdata[mdata.obs['GSE'].isin(target_set)]
    
    # redo harmony in the subset
    import scanpy.external as sce
    sc.pp.pca(mdata["gex"], n_comps=50)
    sc.pp.neighbors(mdata["gex"], n_neighbors = 50)
    sc.tl.umap(mdata["gex"], min_dist=0.5, spread=1.0)
    sce.pp.harmony_integrate(mdata["gex"], key=["GSE", 'sample_id'],  basis='X_pca',
                          theta = 3, lamb = 1,  sigma=0.1, nclust = 50, tau=1)


### If test group is isolated
if_test_isolated = True
if if_test_isolated == True:
    test_ids = ['GSE293883']
else:
    train_frac = 0.7
    
### If use x_umap of X_umap_harmony
Batch_corrected = True

mdata    

MuData object with n_obs × n_vars = 125524 × 3001
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'cloned', 'condition', 'sample_id', 'set', 'state'
  2 modalities
    airr:	125524 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition', 'GSE'
      obsm:	'airr', 'chain_indices'
    gex:	125524 x 3001
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'

# data pre-vision

In [77]:
mdata['gex'].obs['tissue'].isna().sum()

np.int64(6774)

In [78]:
# mdata['gex'].obs['Tissue_group'] = mdata['gex'].obs['tissue'].apply(lambda x: x if (x == 'CNS' or x =='Spleen') else 'else')

mdata['gex'].obs['Tissue_group'] = mdata['gex'].obs['tissue'].apply(lambda x: x if (x == 'CNS') else 'non-CNS')
mdata['gex'].obs['Tissue_group'].value_counts()

Tissue_group
non-CNS    69141
CNS        49609
Name: count, dtype: int64

# Subsets

#### Cell type subset

In [79]:
# print(mdata.obs['condition'].value_counts())
# print(mdata.obs['cell_type'].value_counts())
# print(mdata['gex'].obs['tissue'].value_counts())

In [80]:
# mdata = mdata[mdata.obs['condition'].isin(['EAE'])]
# mdata = mdata[~mdata.obs['cell_type'].isin(['CD8'])]

# Suppose mdata['gex'].obs has columns 'state' and 'cell_type'
# cols = ['state', 'cell_type']
# mask = mdata['gex'].obs[cols].notna().all(axis=1)
# mdata = mdata[mask].copy()

In [81]:
mdata

MuData object with n_obs × n_vars = 125524 × 3001
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'cloned', 'condition', 'sample_id', 'set', 'state'
  2 modalities
    airr:	125524 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition', 'GSE'
      obsm:	'airr', 'chain_indices'
    gex:	125524 x 3001
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE', 'Tissue_group'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'

#### only cloned TCRs

In [82]:
min_clone = 1
cloned_mask = mdata['airr'].obs['clone_id_size'].fillna(0).astype('int') > min_clone
mdata_cloned = mdata[cloned_mask].copy()
mdata_single = mdata[~cloned_mask].copy()
mdata_cloned['airr'].obs['GSE'].value_counts()

GSE
GSE182747    26215
GSE188320    15885
LEE           6310
GSE156718     3182
GSE293883     2483
GSE178085      511
Name: count, dtype: int64

#### select only one cell from each clonotype

In [83]:
# Randomly keep one cell per clone_id within each GSE subset
airr = mdata_cloned['airr']
# airr = mdata['airr']
print(f"Original number of cells: {airr.n_obs}")

# Store indices to keep
oneCell_per_clone_indices = []

# Group by GSE
for gse in airr.obs['GSE'].unique():
    # Get cells for this GSE
    gse_mask = airr.obs['GSE'] == gse
    gse_obs = airr.obs[gse_mask]
    
    print(f"\nGSE: {gse} - Original cells: {len(gse_obs)}")
    
    # For each clone_id in this GSE, randomly select one cell
    for clone_id in gse_obs['clone_id'].dropna().unique():
        clone_cells = gse_obs[gse_obs['clone_id'] == clone_id]
        if len(clone_cells) > 1:
        # Randomly sample one cell from this clone
            selected_idx = clone_cells.sample(n=1, random_state=42).index[0]
            oneCell_per_clone_indices.append(selected_idx)
        else:
            oneCell_per_clone_indices.append(clone_cells.index[0])
    
    print(f"GSE: {gse} - After sampling: {len(gse_obs['clone_id'].unique())} cells (one per clone)")

# Create subset with selected cells
mdata_cloned_oneCell = mdata[oneCell_per_clone_indices].copy()


Original number of cells: 54586

GSE: LEE - Original cells: 6310
GSE: LEE - After sampling: 1689 cells (one per clone)

GSE: GSE156718 - Original cells: 3182
GSE: GSE156718 - After sampling: 1111 cells (one per clone)

GSE: GSE182747 - Original cells: 26215
GSE: GSE182747 - After sampling: 3542 cells (one per clone)

GSE: GSE178085 - Original cells: 511
GSE: GSE178085 - After sampling: 2 cells (one per clone)

GSE: GSE188320 - Original cells: 15885
GSE: GSE188320 - After sampling: 2597 cells (one per clone)

GSE: GSE293883 - Original cells: 2483
GSE: GSE293883 - After sampling: 652 cells (one per clone)


## select subset

In [84]:
single_per_clone = list(mdata_single.obs['GSE'].index) + oneCell_per_clone_indices
mdata_single_withCloned = mdata[single_per_clone].copy()

# mdata = mdata_single
mdata

MuData object with n_obs × n_vars = 125524 × 3001
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'cloned', 'condition', 'sample_id', 'set', 'state'
  2 modalities
    airr:	125524 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition', 'GSE'
      obsm:	'airr', 'chain_indices'
    gex:	125524 x 3001
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE', 'Tissue_group'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'

# embed TCR AA into vector

In [85]:
# beta chain only
# tcr_aa_obs = ['VDJ_1_cdr3_aa',]
# tcr_cat_features = ['VDJ_1_j_call', 'VDJ_1_v_call']

# both chains
tcr_aa_obs = ['VDJ_1_cdr3_aa', 'VJ_1_cdr3_aa']
tcr_cat_features = ['VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call']

tcr_num_features = []

In [86]:
# Atchley factors for the 20 amino acids
atchley_factors = {
    'A': [ 0.591, -1.302, -0.733,  1.570, -0.146],  # Alanine
    'R': [ 1.538,  0.055,  1.502,  0.440,  2.897],  # Arginine
    'N': [ 0.945,  0.828,  1.299, -0.169,  0.933],  # Asparagine
    'D': [ 1.050,  0.302, -3.656, -0.259, -3.242],  # Aspartic acid
    'C': [-1.343,  0.465, -0.862, -1.020, -0.255],  # Cysteine
    'Q': [ 0.931,  0.179, -3.005, -0.503, -1.853],  # Glutamine
    'E': [ 1.357,  0.113, -3.242, -0.339, -2.192],  # Glutamic acid
    'G': [ 0.384,  1.652,  1.330,  1.045,  2.064],  # Glycine
    'H': [ 0.336, -0.417, -1.673, -1.474, -0.078],  # Histidine
    'I': [-1.239, -0.547,  2.131,  0.393,  0.816],  # Isoleucine
    'L': [-1.019, -0.987, -1.505,  1.266, -0.912],  # Leucine
    'K': [ 1.831, -0.561,  0.533, -0.277,  1.648],  # Lysine
    'M': [-0.663, -1.524,  2.219, -1.005,  1.212],  # Methionine
    'F': [-1.006, -0.590,  1.891, -0.397,  0.412],  # Phenylalanine
    'P': [ 0.189,  2.081, -1.628,  0.421, -1.392],  # Proline
    'S': [ 0.228,  1.399, -4.760,  0.670, -2.647],  # Serine
    'T': [ 0.032,  2.213, -1.455,  0.311, -0.259],  # Threonine
    'W': [-0.595,  0.009,  0.672, -2.128, -0.184],  # Tryptophan
    'Y': [ 0.260,  0.830,  3.097, -0.838,  1.512],  # Tyrosine
    'V': [-1.337, -0.279, -0.544,  1.242, -1.262],  # Valine
}


In [87]:
# Vectorize each TCR amino acid column and compute composition & length
for aa_col in tcr_aa_obs:
    # 1. Sequence length
    # Only keep cells with both alpha and beta chains

    lengths = TCR_embedings.compute_sequence_lengths(mdata, aa_col)
    lengths_mask = (lengths> 5) & (lengths< 30)
    len_key = f'{aa_col}_length'
    mdata = mdata[lengths_mask]
    mdata.obs[len_key] = lengths[lengths_mask]
    print(f"Stored {aa_col} lengths in mdata.obs['{len_key}'] with shape {mdata.obs[len_key].shape}")
    
    # 2. Atchley factor encoding
    encoded = TCR_embedings.vectorize_tcr_column(mdata, aa_col, atchley_factors)
    key_name = f'X_{aa_col}_atchley'
    mdata.obsm[key_name] = encoded
    print(f"Stored {aa_col} Atchley vectors in mdata.obsm['{key_name}'] with shape {encoded.shape}")
    
    # 3. AA composition (percentage of each of 20 AAs)
    aa_comp = TCR_embedings.compute_aa_composition_matrix(mdata, aa_col)
    comp_key = f'X_{aa_col}_composition'
    mdata.obsm[comp_key] = aa_comp
    print(f"Stored {aa_col} AA composition in mdata.obsm['{comp_key}'] with shape {aa_comp.shape}\n")
    

Stored VDJ_1_cdr3_aa lengths in mdata.obs['VDJ_1_cdr3_aa_length'] with shape (120682,)
Stored VDJ_1_cdr3_aa Atchley vectors in mdata.obsm['X_VDJ_1_cdr3_aa_atchley'] with shape (120682, 140)
Stored VDJ_1_cdr3_aa AA composition in mdata.obsm['X_VDJ_1_cdr3_aa_composition'] with shape (120682, 20)

Stored VJ_1_cdr3_aa lengths in mdata.obs['VJ_1_cdr3_aa_length'] with shape (106241,)
Stored VJ_1_cdr3_aa Atchley vectors in mdata.obsm['X_VJ_1_cdr3_aa_atchley'] with shape (106241, 115)
Stored VJ_1_cdr3_aa AA composition in mdata.obsm['X_VJ_1_cdr3_aa_composition'] with shape (106241, 20)



In [88]:
mdata

MuData object with n_obs × n_vars = 106241 × 3001
  obs:	'GSE', 'GSM', 'VDJ_1_cdr3_aa', 'VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_cdr3_aa', 'VJ_1_j_call', 'VJ_1_v_call', 'cell_type', 'cloned', 'condition', 'sample_id', 'set', 'state', 'VDJ_1_cdr3_aa_length', 'VJ_1_cdr3_aa_length'
  obsm:	'X_VDJ_1_cdr3_aa_atchley', 'X_VDJ_1_cdr3_aa_composition', 'X_VJ_1_cdr3_aa_atchley', 'X_VJ_1_cdr3_aa_composition'
  2 modalities
    airr:	106241 x 0
      obs:	'sample_id', 'receptor_type', 'receptor_subtype', 'chain_pairing', 'clone_id', 'clone_id_size', 'clonal_expansion', 'sample', 'tissue', 'condition', 'GSE'
      obsm:	'airr', 'chain_indices'
    gex:	106241 x 3001
      obs:	'sample_id', 'date', 'tissue', 'sample', 'mouse_BC', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'CD4score', 'CD8score', 'Tregscore', 'Th17score', 'cell_type', 'IFN_stimscore', 'Activationscore', 'Exhaustscore', 'Mem_Naivescore', 'state', 'TCR_clonotype_frequency', 'condition', 'batch', 'leiden', 'GSE', 'Tissue_group'
      var:	'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
      uns:	'GSE_colors', 'X_umap_harmony', 'cell_type_colors', 'leiden', 'neighbors', 'neighbors_harmony', 'pca', 'state_colors', 'tissue_colors', 'umap'
      obsm:	'X_pca', 'X_pca_harmony', 'X_umap', 'X_umap_harmony'
      varm:	'PCs'
      obsp:	'connectivities', 'distances', 'neighbors_harmony_connectivities', 'neighbors_harmony_distances'

In [89]:
# One-hot encode all categorical TCR features and concatenate
cat_encoded_list = []
for cat_col in tcr_cat_features:
    encoded = TCR_embedings.onehot_encode_categorical(mdata, cat_col)
    # cat_encoded_list.append(encoded)
    mdata.obsm[cat_col] = encoded

# Concatenate all one-hot encoded vectors
# tcr_cat_onehot = np.concatenate(cat_encoded_list, axis=1)


VDJ_1_j_call: 12 unique categories + 1 unknown = 13 dimensions
VDJ_1_v_call: 23 unique categories + 1 unknown = 24 dimensions
VJ_1_j_call: 42 unique categories + 1 unknown = 43 dimensions
VJ_1_v_call: 105 unique categories + 1 unknown = 106 dimensions


In [99]:
# For each GSE, count unique categories in TCR columns
tcr_columns = ['VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call']
cdr3_columns = ['VDJ_1_cdr3_aa', 'VJ_1_cdr3_aa']

for gse in sorted(mdata.obs['GSE'].unique()):
    gse_mask = mdata.obs['GSE'] == gse
    print(f"\nGSE: {gse}")
    print(f"  Number of cells: {gse_mask.sum()}")
    for col in tcr_columns:
        unique_cats = mdata.obs.loc[gse_mask, col].dropna().unique()
        n_unique = len(unique_cats)
        print(f"  {col}: {n_unique} unique categories")
    for col in cdr3_columns:
        unique_seqs = mdata.obs.loc[gse_mask, col].dropna().unique()
        n_unique = len(unique_seqs)
        print(f"  {col}: {n_unique} unique sequences")



GSE: GSE156718
  Number of cells: 9134
  VDJ_1_j_call: 12 unique categories
  VDJ_1_v_call: 22 unique categories
  VJ_1_j_call: 41 unique categories
  VJ_1_v_call: 88 unique categories
  VDJ_1_cdr3_aa: 6677 unique sequences
  VJ_1_cdr3_aa: 5908 unique sequences

GSE: GSE178085
  Number of cells: 501
  VDJ_1_j_call: 1 unique categories
  VDJ_1_v_call: 1 unique categories
  VJ_1_j_call: 1 unique categories
  VJ_1_v_call: 1 unique categories
  VDJ_1_cdr3_aa: 1 unique sequences
  VJ_1_cdr3_aa: 1 unique sequences

GSE: GSE182747
  Number of cells: 30203
  VDJ_1_j_call: 12 unique categories
  VDJ_1_v_call: 22 unique categories
  VJ_1_j_call: 41 unique categories
  VJ_1_v_call: 90 unique categories
  VDJ_1_cdr3_aa: 9726 unique sequences
  VJ_1_cdr3_aa: 8331 unique sequences

GSE: GSE188320
  Number of cells: 16846
  VDJ_1_j_call: 12 unique categories
  VDJ_1_v_call: 21 unique categories
  VJ_1_j_call: 42 unique categories
  VJ_1_v_call: 89 unique categories
  VDJ_1_cdr3_aa: 4092 unique seque

## concate TCR features

In [92]:
arrs_tcr = []
for key, value in mdata.obsm.items():
    arrs_tcr.append(value)

In [93]:
view_tcr = np.concatenate(arrs_tcr, axis=1)
print(view_tcr.shape)

# Add chain length
for chain in tcr_aa_obs:
    view_tcr = np.concatenate([view_tcr, mdata.obs[chain + '_length'].to_numpy().reshape(-1, 1)],  axis=1)

# Add clone size
view_tcr = np.concatenate([view_tcr, mdata['airr'].obs['clone_id_size'].to_numpy().reshape(-1, 1)],  axis=1)
    
print(view_tcr.shape)

(106241, 483)
(106241, 486)


In [94]:
# Perform Canonical Correlation Analysis separately on train and test sets
view_gene = mdata['gex'].X.toarray()
# view_gene = mdata['gex'].obsm['X_pca_harmony']

scaler = StandardScaler()
view_tcr = scaler.fit_transform(view_tcr)
view_gene = scaler.fit_transform(view_gene)

In [95]:
mdata.obsm['tcr_embs'] = view_tcr

In [ ]:
import anndata as ad
ad.settings.allow_write_nullable_strings = True
mdata.write(filename+'_atchleyEmbs_allcells.h5mu')